In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd


### 1. Paths

In [ ]:
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()

INPUT_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "metrics_labeling.csv"
)

TRAIN_OUTPUT_PATH = (
    PROJECT_ROOT / "data" / "processed" / "train_dataset.csv"
)
VAL_OUTPUT_PATH = (
    PROJECT_ROOT / "data" / "processed" / "val_dataset.csv"
)
TEST_OUTPUT_PATH = (
    PROJECT_ROOT / "data" / "processed" / "test_dataset.csv"
)
SPLIT_SUMMARY_PATH = (
    PROJECT_ROOT / "reports" / "split_summary.csv"
)


### 2. Split configuration

- `SPLIT_RATIOS` are target shares of **total duration** (seconds), not row count or run count, so a handful of long runs cannot silently dominate a split.
- `MIN_RUN_DURATION_SECONDS` excludes runs too short to be assigned reliably.
- The split unit is the whole **run** (`GROUP_COLUMN`): a run's rows always stay together in the same split.

In [ ]:
# Target share of TOTAL DURATION (in seconds) assigned to each split.
# Train gets the earliest data, val the following slice, test the most
# recent slice -- this mirrors how the model will be used in production
# (trained on the past, evaluated on data that comes after it).
SPLIT_RATIOS = {
    "train": 0.70,
    "val": 0.15,
    "test": 0.15,
}

# A run shorter than this is excluded from the split entirely: with so
# few rows it cannot be meaningfully assigned to one split without
# skewing it, and it contributes very little signal either way.
MIN_RUN_DURATION_SECONDS = 900  # 15 minutes

GROUP_COLUMN = "run_id"
MACHINE_COLUMN = "machine_id"
TIME_COLUMN = "timestamp"
TARGET_COLUMN = "slowdown_in_5min"


### 3. Build one row per run (duration, size, class balance)

This is the table the split decision is actually made on: one line per `run_id`, not per timestamp.

In [ ]:
def build_run_table(df: pd.DataFrame) -> pd.DataFrame:
    run_table = (
        df.groupby([MACHINE_COLUMN, GROUP_COLUMN])
        .agg(
            start_time=(TIME_COLUMN, "min"),
            end_time=(TIME_COLUMN, "max"),
            n_rows=(GROUP_COLUMN, "size"),
            n_positive=(TARGET_COLUMN, "sum"),
        )
        .reset_index()
    )

    run_table["duration_seconds"] = (
        run_table["end_time"] - run_table["start_time"]
    ).dt.total_seconds()

    run_table["positive_rate"] = (
        run_table["n_positive"] / run_table["n_rows"]
    )

    return run_table


### 4. Assign each run to train / val / test, chronologically per machine

Rules that must never be broken:
- A run is an atomic unit: all its rows go to the same split.
- The assignment is chronological **per machine** (sorted by `start_time`), never shuffled: the earliest runs become train, the next slice becomes val, the most recent slice becomes test.
- Cut points are based on cumulative **duration**, not on row count or run count.

In [ ]:
def assign_runs_to_splits(run_table: pd.DataFrame) -> pd.DataFrame:
    train_boundary = SPLIT_RATIOS["train"]
    val_boundary = SPLIT_RATIOS["train"] + SPLIT_RATIOS["val"]

    assignments = []

    for machine_id, group in run_table.groupby(MACHINE_COLUMN):
        group = group.sort_values("start_time").reset_index(drop=True)
        total_duration = group["duration_seconds"].sum()

        cumulative = 0.0
        for _, run_row in group.iterrows():
            midpoint = cumulative + run_row["duration_seconds"] / 2
            cumulative += run_row["duration_seconds"]

            fraction = midpoint / total_duration if total_duration > 0 else 0.0

            if fraction <= train_boundary:
                split = "train"
            elif fraction <= val_boundary:
                split = "val"
            else:
                split = "test"

            assignments.append(
                {
                    MACHINE_COLUMN: machine_id,
                    GROUP_COLUMN: run_row[GROUP_COLUMN],
                    "split": split,
                }
            )

    return pd.DataFrame(assignments)


### 5. Validate the split (no leakage, correct chronological order)

Two checks that must both pass before anything is saved:
1. No `run_id` appears in more than one split.
2. On every machine, all train timestamps come before all val timestamps, which come before all test timestamps.

In [ ]:
def validate_split(df: pd.DataFrame) -> None:
    # 1. A run must never appear in more than one split.
    runs_per_split_count = df.groupby(GROUP_COLUMN)["split"].nunique()
    leaking_runs = runs_per_split_count[runs_per_split_count > 1]
    if len(leaking_runs) > 0:
        raise AssertionError(
            f"Data leakage: runs assigned to multiple splits: "
            f"{leaking_runs.index.tolist()}"
        )

    # 2. Within each machine, splits must be chronologically ordered.
    split_order = {"train": 0, "val": 1, "test": 2}
    for machine_id, machine_df in df.groupby(MACHINE_COLUMN):
        present_splits = [
            split_name
            for split_name in ["train", "val", "test"]
            if split_name in machine_df["split"].unique()
        ]

        boundaries = {}
        for split_name in present_splits:
            split_rows = machine_df.loc[machine_df["split"] == split_name, TIME_COLUMN]
            boundaries[split_name] = (split_rows.min(), split_rows.max())

        ordered = sorted(present_splits, key=lambda s: split_order[s])
        for earlier, later in zip(ordered, ordered[1:]):
            if boundaries[earlier][1] > boundaries[later][0]:
                raise AssertionError(
                    f"Temporal overlap on machine {machine_id}: "
                    f"{earlier} ends after {later} starts"
                )

    print(
        "Validation passed: no run appears in two splits, "
        "and train < val < test chronologically on every machine."
    )


### 6. Build a human-readable split summary

In [ ]:
def build_split_summary(df: pd.DataFrame, excluded_runs: pd.DataFrame) -> pd.DataFrame:
    summary_rows = []
    for split_name in ["train", "val", "test"]:
        split_df = df[df["split"] == split_name]
        summary_rows.append(
            {
                "split": split_name,
                "n_rows": len(split_df),
                "n_runs": split_df[GROUP_COLUMN].nunique(),
                "n_machines": split_df[MACHINE_COLUMN].nunique(),
                "positive_rate": round(split_df[TARGET_COLUMN].mean(), 4)
                if len(split_df) > 0 else float("nan"),
                "start_time": split_df[TIME_COLUMN].min(),
                "end_time": split_df[TIME_COLUMN].max(),
            }
        )

    summary_rows.append(
        {
            "split": "excluded_short_runs",
            "n_rows": excluded_runs["n_rows"].sum() if len(excluded_runs) else 0,
            "n_runs": len(excluded_runs),
            "n_machines": excluded_runs[MACHINE_COLUMN].nunique() if len(excluded_runs) else 0,
            "positive_rate": float("nan"),
            "start_time": pd.NaT,
            "end_time": pd.NaT,
        }
    )

    return pd.DataFrame(summary_rows)


### 7. Main split function

Loads `metrics_labeling.csv`, builds the run table, excludes short runs, assigns train/val/test, validates, then saves the three datasets plus a summary report. Nothing is written to disk unless validation passes.

In [ ]:
def temporal_split() -> None:
    if not INPUT_PATH.exists():
        raise FileNotFoundError(f"Input file not found: {INPUT_PATH}")

    print(f"Loading dataset: {INPUT_PATH}")
    df = pd.read_csv(INPUT_PATH)
    df[TIME_COLUMN] = pd.to_datetime(df[TIME_COLUMN], errors="coerce")

    if df[TIME_COLUMN].isna().any():
        raise ValueError(
            f"{df[TIME_COLUMN].isna().sum()} rows have an unparsable {TIME_COLUMN}"
        )

    required_columns = [MACHINE_COLUMN, GROUP_COLUMN, TIME_COLUMN, TARGET_COLUMN]
    missing_columns = [c for c in required_columns if c not in df.columns]
    if missing_columns:
        raise ValueError(f"Missing required columns: {missing_columns}")

    initial_rows = len(df)

    # Step 1: one row per run, with duration and class balance.
    run_table = build_run_table(df)

    # Step 2: exclude runs that are too short to split reliably.
    short_runs = run_table[run_table["duration_seconds"] < MIN_RUN_DURATION_SECONDS]
    usable_runs = run_table[run_table["duration_seconds"] >= MIN_RUN_DURATION_SECONDS]

    # Step 3: assign each usable run to train/val/test, chronologically,
    # independently per machine.
    split_assignment = assign_runs_to_splits(usable_runs)

    # Step 4: merge the assignment back to row level.
    df_split = df.merge(split_assignment, on=[MACHINE_COLUMN, GROUP_COLUMN], how="inner")

    # Step 5: validate before saving anything.
    validate_split(df_split)

    # Step 6: save the three datasets.
    for split_name, output_path in [
        ("train", TRAIN_OUTPUT_PATH),
        ("val", VAL_OUTPUT_PATH),
        ("test", TEST_OUTPUT_PATH),
    ]:
        output_path.parent.mkdir(parents=True, exist_ok=True)
        df_split[df_split["split"] == split_name].drop(columns=["split"]).to_csv(
            output_path, index=False
        )

    # Step 7: save a human-readable summary report.
    split_summary = build_split_summary(df_split, short_runs)
    SPLIT_SUMMARY_PATH.parent.mkdir(parents=True, exist_ok=True)
    split_summary.to_csv(SPLIT_SUMMARY_PATH, index=False)

    print("\nTemporal split completed.")
    print(f"Initial rows: {initial_rows}")
    print(f"Runs excluded (duration < {MIN_RUN_DURATION_SECONDS}s): {len(short_runs)}")
    print(f"Rows used in the split: {len(df_split)}")
    print()
    print(split_summary.to_string(index=False))
    print()
    print(f"Train saved to: {TRAIN_OUTPUT_PATH}")
    print(f"Val saved to:   {VAL_OUTPUT_PATH}")
    print(f"Test saved to:  {TEST_OUTPUT_PATH}")
    print(f"Summary saved to: {SPLIT_SUMMARY_PATH}")


In [ ]:
if __name__ == "__main__":
    temporal_split()